# Shift-Share Decomposition: Initial & Persistent ADR%

Decomposes month-over-month ADR% changes into **composition effects** (case mix shifted) vs **rate effects** (denial rates changed within categories) across 5 factors.

**Factors**: IP_type, market, AHRQ dx category, LOS bucket, hospital group  
**Population**: M&R FFS  
**Period**: 202301-202605 (excl. 202403-202404)  
**Source**: `tmp_1m.knd_loc_shiftshare_base`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL

engine = create_engine(URL(
    account = "UHG-UHGDWAAS",
    user = "KHANG.NGUYEN@UHC.COM",
    authenticator = "externalbrowser",
    role = "AZU_SDRP_VING_PRD_DEVELOPER_ROLE",
    warehouse = "VING_PRD_MNR_HCE_DATAINFRA_WH",
    database = "VING_PRD_TREND_DB",
    schema = "TMP_1M"
))

df = pd.read_sql("""
    select factor_name, factor_value, hce_admit_month
        , case_count, initial_adr_cnt, persistent_adr_cnt
    from tmp_1m.knd_loc_shiftshare_base
    order by factor_name, factor_value, hce_admit_month
""", engine)

df.columns = df.columns.str.lower()
print(df.shape)
df.head()

c:\Users\knguy139\Documents\Projects\.venv\Lib\site-packages\snowflake\connector\vendored\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(
 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/db05faca-c82a-4b9d-b9c5-0f64b6755421/saml2?SAMLRequest=lVJdb9owFP0rkfec2AkJLRZQ8TE61I%2FRknbSXibHdsAisYPtEPj3cwJI3UMr7cGSZZ9zz7n33OHdsSy8A9dGKDkCYYCAxyVVTMjNCLylC%2F8WeMYSyUihJB%2BBEzfgbjw0pCwqPKntVr7yfc2N9VwhaXD7MQK1llgRIwyWpOQGW4rXk6dHHAUIE2O4tk4OXCjMCKe1tbbCEDZNEzS9QOkNjBBCEA2gQ7WQb%2BCDRPW1RqWVVVQVV8rR9fSJRAhR3Eo4hFNYXYhTIc8j%2BEolO4MM%2FpGmK3%2F1c50Cb3LtbqakqUuu11wfBOVvr49nA8Y5qLcb3x3WEGL%2BbElgpGryguw4VWVVW1czcDeYcwYLtRFuUsv5CFQ7wZ6elbzPFlN%2BIjflPn5IHtLDy8s6fb%2BfEXGsq0k43aNfg%2B%2B79ZIC7%2F2aa9TmujSm5kvZpmndE4r6Prr1e2EaDnDSw2Ec9JP4N%2FDmLk0hie2YV8udj6AUVCujcqtkISTvXLIMJTmhxKe3EfHjbMD8bEATH%2BX9OOvfJEkchbDNLALnvcGdET3%2Bv2kM4UfuZQGfXSbL%2BUoVgp68hdIlsZ9HFgZh9yKYn3dQzEsiigljmhvjoisK1cw0J9btudU1B3B8Vv1308d%2FAQ%3D%3D&RelayState=ver%3A3-hint%3A260092298610362-ETMsDgAAAaBZYhKeABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEDtct1%2F%2BF3dr2f3z90ZPPL8AAACQ

,factor_name,factor_value,hce_admit_month,case_count,initial_adr_cnt,persistent_adr_cnt
0,ahrq_dx,BLOOD & BLOOD FORM ORGANS,202301,2099,584,411
1,ahrq_dx,BLOOD & BLOOD FORM ORGANS,202302,1959,570,375
2,ahrq_dx,BLOOD & BLOOD FORM ORGANS,202303,2196,630,419
3,ahrq_dx,BLOOD & BLOOD FORM ORGANS,202304,1965,557,384
4,ahrq_dx,BLOOD & BLOOD FORM ORGANS,202305,2118,657,423


In [ ]:
# overall monthly ADR% (from ip_type factor since totals are consistent across factors)
overall = (
    df
    .query("factor_name == 'ip_type'")
    .groupby("hce_admit_month")
    .agg(case_count = ("case_count", "sum"),
         initial_adr_cnt = ("initial_adr_cnt", "sum"),
         persistent_adr_cnt = ("persistent_adr_cnt", "sum"))
    .reset_index()
    .assign(
        initial_adr_rate = lambda x: x.initial_adr_cnt / x.case_count,
        persistent_adr_rate = lambda x: x.persistent_adr_cnt / x.case_count
    )
)
overall.head()

In [ ]:
def shift_share(df_factor, outcome_col):
    months = sorted(df_factor["hce_admit_month"].unique())
    results = []

    for i in range(1, len(months)):
        m_prev, m_curr = months[i - 1], months[i]

        prev = df_factor.query("hce_admit_month == @m_prev").set_index("factor_value")
        curr = df_factor.query("hce_admit_month == @m_curr").set_index("factor_value")

        # align on factor levels present in either period
        all_levels = prev.index.union(curr.index)
        prev = prev.reindex(all_levels, fill_value = 0)
        curr = curr.reindex(all_levels, fill_value = 0)

        total_prev = prev["case_count"].sum()
        total_curr = curr["case_count"].sum()

        if total_prev == 0 or total_curr == 0:
            continue

        share_prev = prev["case_count"] / total_prev
        share_curr = curr["case_count"] / total_curr
        rate_prev = np.where(prev["case_count"] > 0, prev[outcome_col] / prev["case_count"], 0)
        rate_curr = np.where(curr["case_count"] > 0, curr[outcome_col] / curr["case_count"], 0)

        delta_share = share_curr - share_prev
        delta_rate = rate_curr - rate_prev

        composition = (delta_share * rate_prev).sum()
        rate_effect = (share_prev * delta_rate).sum()
        interaction = (delta_share * delta_rate).sum()

        results.append({
            "month_from": m_prev,
            "month_to": m_curr,
            "composition_bps": round(composition * 10000, 2),
            "rate_bps": round(rate_effect * 10000, 2),
            "interaction_bps": round(interaction * 10000, 2),
            "total_change_bps": round((composition + rate_effect + interaction) * 10000, 2)
        })

    return pd.DataFrame(results)

In [ ]:
# run shift-share for all 5 factors x 2 outcomes
factors = df["factor_name"].unique()
outcomes = {"initial_adr_cnt": "Initial ADR", "persistent_adr_cnt": "Persistent ADR"}

all_results = []
for factor in factors:
    df_f = df.query("factor_name == @factor")
    for outcome_col, outcome_label in outcomes.items():
        result = shift_share(df_f, outcome_col)
        result = result.assign(factor_name = factor, outcome = outcome_label)
        all_results.append(result)

decomp = pd.concat(all_results, ignore_index = True)
decomp = decomp[["outcome", "factor_name", "month_from", "month_to",
                  "composition_bps", "rate_bps", "interaction_bps", "total_change_bps"]]

print(decomp.shape)
decomp.head(10)

In [ ]:
# sanity check: total_change_bps should match actual rate change
check = (
    decomp
    .query("factor_name == 'ip_type' and outcome == 'Initial ADR'")
    .assign(cumulative_bps = lambda x: x.total_change_bps.cumsum())
)

actual_start = overall.iloc[0]["initial_adr_rate"]
actual_end = overall.iloc[-1]["initial_adr_rate"]
actual_change_bps = round((actual_end - actual_start) * 10000, 2)
decomp_change_bps = round(check["total_change_bps"].sum(), 2)

print(f"Actual change: {actual_change_bps} bps")
print(f"Decomp sum:    {decomp_change_bps} bps")
print(f"Match: {abs(actual_change_bps - decomp_change_bps) < 1}")

In [ ]:
# cumulative summary table: total bps by factor x outcome
summary = (
    decomp
    .groupby(["outcome", "factor_name"])
    .agg(
        composition_bps = ("composition_bps", "sum"),
        rate_bps = ("rate_bps", "sum"),
        interaction_bps = ("interaction_bps", "sum"),
        total_change_bps = ("total_change_bps", "sum")
    )
    .round(1)
    .reset_index()
    .sort_values(["outcome", "total_change_bps"], ascending = [True, False])
)
summary

In [ ]:
# plot 1: overall ADR% trend
fig, ax = plt.subplots(figsize = (14, 5))
ax.plot(overall["hce_admit_month"], overall["initial_adr_rate"] * 100, marker = "o", ms = 3, label = "Initial ADR%")
ax.plot(overall["hce_admit_month"], overall["persistent_adr_rate"] * 100, marker = "s", ms = 3, label = "Persistent ADR%")
ax.set_xlabel("Admit Month")
ax.set_ylabel("ADR Rate (%)")
ax.set_title("Monthly Initial & Persistent ADR% - M&R FFS (excl. 202403-04)")
ax.legend()
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.1f%%"))
for i, label in enumerate(ax.get_xticklabels()):
    if i % 3 != 0:
        label.set_visible(False)
ax.tick_params(axis = "x", rotation = 45)
plt.tight_layout()
plt.show()

In [ ]:
# plot 2: per-factor stacked bars - composition vs rate vs interaction (Initial ADR only)
fig, axes = plt.subplots(5, 1, figsize = (14, 20), sharex = True)
factor_order = ["ip_type", "market", "ahrq_dx", "los_bucket", "hospital_group"]

for ax, factor in zip(axes, factor_order):
    fd = decomp.query("factor_name == @factor and outcome == 'Initial ADR'").copy()
    x = range(len(fd))
    ax.bar(x, fd["composition_bps"], label = "Composition", color = "#4C72B0", alpha = 0.8)
    ax.bar(x, fd["rate_bps"], bottom = fd["composition_bps"], label = "Rate", color = "#DD8452", alpha = 0.8)
    ax.bar(x, fd["interaction_bps"],
           bottom = fd["composition_bps"] + fd["rate_bps"],
           label = "Interaction", color = "#CCCCCC", alpha = 0.8)
    ax.axhline(0, color = "black", linewidth = 0.5)
    ax.set_ylabel("bps")
    ax.set_title(f"Initial ADR% Decomposition: {factor}")
    if ax == axes[0]:
        ax.legend(loc = "upper left")

axes[-1].set_xticks(range(len(fd)))
axes[-1].set_xticklabels(fd["month_to"], rotation = 45, ha = "right", fontsize = 7)
axes[-1].set_xlabel("Month")

plt.tight_layout()
plt.show()

In [ ]:
# plot 3: cumulative waterfall - total composition vs rate bps by factor (Initial ADR)
init_summary = summary.query("outcome == 'Initial ADR'").copy()

fig, ax = plt.subplots(figsize = (10, 6))
x = range(len(init_summary))
w = 0.35
ax.bar([i - w / 2 for i in x], init_summary["composition_bps"], w, label = "Composition", color = "#4C72B0")
ax.bar([i + w / 2 for i in x], init_summary["rate_bps"], w, label = "Rate", color = "#DD8452")
ax.set_xticks(list(x))
ax.set_xticklabels(init_summary["factor_name"], rotation = 30, ha = "right")
ax.set_ylabel("Cumulative bps (202301 to 202605)")
ax.set_title("Cumulative Shift-Share Attribution: Initial ADR%")
ax.axhline(0, color = "black", linewidth = 0.5)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# export decomp and summary
output_dir = r"C:\Users\knguy139\Documents\Projects\Data\Output"
decomp.to_csv(f"{output_dir}\\knd_loc_shiftshare_decomp.csv", index = False)
summary.to_csv(f"{output_dir}\\knd_loc_shiftshare_summary.csv", index = False)
print("Exported.")